# LeetCode 159 — Longest Substring with At Most Two Distinct Characters bằng Python

> **Lưu ý: đây là bài Premium** (bài bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở được).
> Danh mục học của bạn xếp bài này vào nhóm **Hard**; theo những gì mình biết, LeetCode hiện gắn nhãn độ khó Medium (trước đây từng là Hard). Dù nhãn là gì, đây là bài **kinh điển nhất** để học kỹ thuật **sliding window** (cửa sổ trượt) có kích thước thay đổi.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lengthOfLongestSubstringTwoDistinct(self, s: str) -> int:

```

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: **substring** (chuỗi con liên tục) khác **subsequence** ra sao, "at most two distinct" nghĩa là gì.
2. Công cụ Python cần dùng: slicing `s[i:j]`, `set`, `dict` đếm tần suất (`get`, `del`), `min(d, key=d.get)`.
3. Bốn cách giải, đi từ trực giác đến khéo léo:
   - **Cách 1a:** Brute force thử **mọi** chuỗi con + `set` — `O(n³)`.
   - **Cách 1b:** Cố định đầu trái, mở rộng sang phải, **dừng sớm** khi có ký tự thứ 3 — `O(n²)`.
   - **Cách 2 (cách chính):** **Sliding window** với `dict` **đếm tần suất**, co cửa sổ bằng `while` — `O(n)`.
   - **Cách 3:** Sliding window với `dict` lưu **vị trí xuất hiện cuối** của mỗi ký tự, `left` **nhảy một phát** tới `min(last) + 1`.
   - **Cách 4 (mẹo sáng tạo):** Cửa sổ **không bao giờ co** (non-shrinking window) — chỉ dùng `if`, dịch `left` đúng 1 bước.
4. Vì sao mỗi cách đúng (invariant — "bất biến", điều luôn đúng sau mỗi bước), độ phức tạp, lỗi hay gặp.
5. Tổng quát hóa sang **k** ký tự khác nhau (LeetCode 340) và các bài cùng họ (904, 3, 992...).

## 1. Đề bài (theo LeetCode)

Given a string `s`, return *the length of the longest* ***substring*** *that contains at most* ***two distinct characters***.

**Example 1**

```text
Input:  s = "eceba"
Output: 3
Explanation: The substring is "ece" which its length is 3.
```

**Example 2**

```text
Input:  s = "ccaabbb"
Output: 5
Explanation: The substring is "aabbb" which its length is 5.
```

**Constraints**

- `1 <= s.length <= 10^5`
- `s` consists of English letters.

### Dịch nhanh

- Cho một chuỗi `s`.
- Tìm **chuỗi con liên tục** dài nhất mà bên trong nó có **tối đa 2 loại ký tự khác nhau** (1 loại cũng được, 2 loại cũng được, 3 loại thì không).
- Chỉ trả về **độ dài** (một số nguyên), không cần trả về chính chuỗi con.
- "English letters" = chữ cái tiếng Anh, gồm cả **chữ hoa và chữ thường** (`'a'` và `'A'` là **hai** ký tự khác nhau). Vậy bảng chữ cái có tối đa 52 ký tự.

## 2. Hiểu đề bằng tay

### 2.1. Substring khác subsequence

- **Substring** (chuỗi con) = một đoạn **liền nhau** của `s`. Với `s = "eceba"`: `"ece"`, `"ceb"`, `"ba"` là substring.
- **Subsequence** (dãy con) = chọn ký tự theo thứ tự nhưng **được bỏ cách quãng**. `"eea"` là subsequence của `"eceba"` nhưng **không** phải substring.

Đề hỏi **substring**, nên đáp án luôn là một đoạn `s[i..j]` liền mạch.

### 2.2. Đếm "số loại ký tự khác nhau" (distinct)

| Chuỗi con | Các ký tự | Loại khác nhau | Hợp lệ (≤ 2)? |
|---|---|---|---|
| `"ece"` | e, c, e | {e, c} → 2 | Có |
| `"eceb"` | e, c, e, b | {e, c, b} → 3 | Không |
| `"aabbb"` | a, a, b, b, b | {a, b} → 2 | Có |
| `"cc"` | c, c | {c} → 1 | Có |

Ô code dưới in vài chuỗi con của `"eceba"` cùng số loại ký tự khác nhau, dùng `len(set(...))` (sẽ giải thích kỹ ở mục 3).

In [1]:
s = "eceba"
for i, j in [(0, 2), (0, 3), (1, 2), (2, 4), (3, 4)]:
    sub = s[i:j + 1]
    distinct = len(set(sub))
    print(f"s[{i}..{j}] = {sub!r:7} -> {distinct} loại, hợp lệ: {distinct <= 2}")

s[0..2] = 'ece'   -> 2 loại, hợp lệ: True
s[0..3] = 'eceb'  -> 3 loại, hợp lệ: False
s[1..2] = 'ce'    -> 2 loại, hợp lệ: True
s[2..4] = 'eba'   -> 3 loại, hợp lệ: False
s[3..4] = 'ba'    -> 2 loại, hợp lệ: True


Nhận xét quan trọng:

- Chuỗi con **dài nhất hợp lệ** của `"eceba"` là `"ece"` (độ dài 3). Không có chuỗi con dài 4 nào hợp lệ: `"eceb"` và `"ceba"` đều có 3 loại.
- Vài **edge case** (trường hợp biên) cần nhớ:
  - `s` chỉ có 1 ký tự (`"a"`) ⇒ đáp án `1`.
  - `s` có ≤ 2 loại ký tự (`"abab"`, `"zzzz"`) ⇒ đáp án là **cả chuỗi**, `len(s)`.
  - `s` mỗi ký tự một khác (`"abcdef"`) ⇒ mọi cặp liền nhau đều hợp lệ, đáp án `2`.

## 3. Nền tảng Python

### 3.1. Chỉ số (index) và slicing `s[i:j]`

- `s[i]` là ký tự ở vị trí `i` (đếm từ `0`).
- `s[i:j]` lấy các ký tự từ `i` **đến trước** `j` — không lấy `j`. Độ dài kết quả là `j - i`.
- Trong notebook này ta hay nói "đoạn `[left..right]`" (lấy **cả hai đầu**). Đoạn đó trong Python là `s[left:right + 1]`, và độ dài là **`right - left + 1`** (đừng quên `+ 1`!).

In [2]:
s = "ccaabbb"
print(s[0], s[2], s[6])          # c a b
print(s[2:7])                    # 'aabbb'  (vị trí 2,3,4,5,6)
left, right = 2, 6
print(s[left:right + 1], "dài", right - left + 1)   # 'aabbb' dài 5
print(len(s))                    # 7

c a b
aabbb
aabbb dài 5
7


### 3.2. `set` — tập hợp, tự bỏ phần tử trùng

`set(chuỗi)` tạo một **tập hợp** các ký tự, mỗi ký tự chỉ giữ **một bản**. Vì vậy `len(set(sub))` = số loại ký tự khác nhau.

Ta cũng có thể tạo `set` rỗng rồi thêm dần bằng `.add(x)` — sẽ dùng ở Cách 1b.

In [3]:
print(set("aabbb"))          # {'a', 'b'}  (thứ tự in có thể khác)
print(len(set("aabbb")))     # 2
print(len(set("eceb")))      # 3

seen = set()
for ch in "cca":
    seen.add(ch)
    print("thêm", ch, "->", sorted(seen), "| số loại:", len(seen))

{'a', 'b'}
2
3
thêm c -> ['c'] | số loại: 1
thêm c -> ['c'] | số loại: 1
thêm a -> ['a', 'c'] | số loại: 2


### 3.3. `dict` đếm tần suất — trái tim của sliding window

Một **dict** (dictionary, "từ điển") lưu cặp `key -> value`. Ở bài này: `key` = ký tự, `value` = **số lần** ký tự đó xuất hiện trong cửa sổ hiện tại.

Ba thao tác cần thuộc:

1. **Thêm một ký tự**: `count[ch] = count.get(ch, 0) + 1`. Hàm `get(ch, 0)` trả về giá trị đang có, hoặc `0` nếu chưa có key `ch` (không bị lỗi `KeyError`).
2. **Bớt một ký tự**: `count[ch] -= 1`.
3. **Xóa key khi đếm về 0**: `if count[ch] == 0: del count[ch]`.

Vì sao bước 3 quan trọng? Vì ta dùng **`len(count)`** làm "số loại ký tự đang có trong cửa sổ". Nếu để lại key có giá trị `0`, `len(count)` sẽ **đếm cả ký tự đã biến mất** ⇒ sai.

### 3.4. `Counter` — tiện nhưng có bẫy

`collections.Counter` là một dict chuyên để đếm: `cnt[ch] += 1` chạy được ngay cả khi chưa có key. Nhưng **`Counter` không tự xóa key khi về 0**, nên `len(cnt)` vẫn bị đếm thừa y như trên. Nếu dùng `Counter`, vẫn phải `del` bằng tay.

In [4]:
count = {}
for ch in "cca":
    count[ch] = count.get(ch, 0) + 1
print(count, "| số loại:", len(count))          # {'c': 2, 'a': 1} | 2

count["c"] -= 1
count["c"] -= 1
print(count, "| số loại:", len(count))          # {'c': 0, 'a': 1} | 2  <- SAI! c đã hết
if count["c"] == 0:
    del count["c"]
print(count, "| số loại:", len(count))          # {'a': 1} | 1  <- đúng


from collections import Counter

cnt = Counter("cca")
print(cnt, len(cnt))        # Counter({'c': 2, 'a': 1}) 2
cnt["c"] -= 2
print(cnt, len(cnt))        # Counter({'a': 1, 'c': 0}) 2  <- vẫn còn key 'c'!
del cnt["c"]
print(cnt, len(cnt))        # Counter({'a': 1}) 1

{'c': 2, 'a': 1} | số loại: 2
{'c': 0, 'a': 1} | số loại: 2
{'a': 1} | số loại: 1
Counter({'c': 2, 'a': 1}) 2
Counter({'a': 1, 'c': 0}) 2
Counter({'a': 1}) 1


### 3.5. `min(d, key=d.get)` — tìm key có value nhỏ nhất

Cách 3 cần trả lời câu hỏi: "Trong các ký tự đang có, ký tự nào **xuất hiện lần cuối sớm nhất**?" Nếu `last = {ký tự: vị trí cuối}`, câu trả lời là:

```python
victim = min(last, key=last.get)
```

Đọc là: "duyệt các **key** của `last`, so sánh chúng bằng `last.get(key)` (tức value), lấy key có value nhỏ nhất".

In [5]:
last = {"c": 1, "a": 3, "b": 4}
victim = min(last, key=last.get)
print("ký tự có vị trí cuối nhỏ nhất:", victim)   # c
print("vị trí đó:", last[victim])                 # 1
print("min(last) không có key=:", min(last))      # 'a' — so sánh theo bảng chữ cái, KHÔNG phải điều ta muốn

ký tự có vị trí cuối nhỏ nhất: c
vị trí đó: 1
min(last) không có key=: a


### 3.6. "Cửa sổ" `[left..right]` và hai con trỏ

Hãy tưởng tượng một **khung kính** đặt trên chuỗi, che từ vị trí `left` đến `right`. Hai biến `left`, `right` gọi là **con trỏ** (pointer) — chúng chỉ là chỉ số, "trỏ" vào hai mép cửa sổ.

- **Mở rộng** cửa sổ: `right += 1` (kéo mép phải sang phải, thêm ký tự `s[right]`).
- **Co** cửa sổ: `left += 1` (kéo mép trái sang phải, bỏ ký tự `s[left]` cũ).

Cả hai con trỏ **chỉ đi về bên phải**, không bao giờ lùi. Ô code dưới chỉ minh họa hình ảnh cửa sổ (chưa phải thuật toán).

### 3.7. `while` khác `if` thế nào?

- `if điều_kiện:` — kiểm tra **một lần**, đúng thì làm **một lần**.
- `while điều_kiện:` — làm **lặp đi lặp lại** cho tới khi điều kiện sai.

Khi co cửa sổ, đôi khi phải bỏ **nhiều** ký tự bên trái mới hết vi phạm (ví dụ bỏ cả `"cc"` mới hết chữ `c`). Vì vậy Cách 2 dùng `while`. Cách 4 lại cố tình dùng `if` — và ta sẽ chứng minh vì sao nó vẫn đúng.

In [6]:
def show_window(s, left, right):
    inside = s[left:right + 1]
    print(" " * left + "[" + inside + "]", f"left={left}, right={right}, dài={right - left + 1}")

s = "ccaabbb"
print(" " + s)
show_window(s, 0, 3)   # mở rộng tới 'ccaa'
show_window(s, 0, 4)   # thêm 'b' -> 3 loại, quá giới hạn
show_window(s, 2, 4)   # co trái 2 bước -> 'aab'
show_window(s, 2, 6)   # mở rộng tiếp -> 'aabbb'


x = 10
if x > 3:
    x -= 3
print("sau if   :", x)    # 7  (chỉ trừ 1 lần)

x = 10
while x > 3:
    x -= 3
print("sau while:", x)    # 1  (trừ tới khi x <= 3)

 ccaabbb
[ccaa] left=0, right=3, dài=4
[ccaab] left=0, right=4, dài=5
  [aab] left=2, right=4, dài=3
  [aabbb] left=2, right=6, dài=5
sau if   : 7
sau while: 1


## 4. Cách 1a — Brute force: thử mọi chuỗi con

Cách người mới nghĩ ra đầu tiên: **liệt kê mọi** đoạn `[i..j]`, đếm số loại ký tự bằng `len(set(...))`, giữ độ dài lớn nhất trong các đoạn hợp lệ.

```text
best = 0
với mỗi i từ 0 tới n-1:
    với mỗi j từ i tới n-1:
        nếu len(set(s[i..j])) <= 2: best = max(best, j - i + 1)
```

Code này chắc chắn đúng (vì xét **hết**), nên ta sẽ giữ nó làm "trọng tài" để kiểm tra các cách nhanh hơn.

In [7]:
def longest_two_distinct_bruteforce(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i, n):
            if len(set(s[i:j + 1])) <= 2:      # cắt + tạo set: tốn O(j - i + 1)
                best = max(best, j - i + 1)
    return best


print(longest_two_distinct_bruteforce("eceba"))     # 3
print(longest_two_distinct_bruteforce("ccaabbb"))   # 5
print(longest_two_distinct_bruteforce("a"))         # 1

3
5
1


### Cách 1a tốn bao nhiêu công?

- Số cặp `(i, j)` là `n(n+1)/2` ≈ `n²/2`.
- Với mỗi cặp, `s[i:j+1]` và `set(...)` phải đọc qua `j - i + 1` ký tự.
- Tổng số ký tự phải đọc = `1·n + 2·(n-1) + ... ` = `n(n+1)(n+2)/6` ≈ `n³/6`.

Vậy **Time `O(n³)`**, **Space `O(n)`** (cho chuỗi cắt ra). Với `n = 10^5`, con số này lớn khủng khiếp:

In [8]:
for n in [10, 100, 1000, 10**5]:
    pairs = n * (n + 1) // 2
    chars = n * (n + 1) * (n + 2) // 6
    print(f"n={n:>6}: số cặp (i,j) = {pairs:>14,} | số ký tự phải đọc ≈ {chars:>22,}")

n=    10: số cặp (i,j) =             55 | số ký tự phải đọc ≈                    220
n=   100: số cặp (i,j) =          5,050 | số ký tự phải đọc ≈                171,700
n=  1000: số cặp (i,j) =        500,500 | số ký tự phải đọc ≈            167,167,000
n=100000: số cặp (i,j) =  5,000,050,000 | số ký tự phải đọc ≈    166,671,666,700,000


## 5. Cách 1b — Cố định trái, mở rộng phải, dừng sớm: `O(n²)`

Cách 1a lãng phí ở hai chỗ:

1. **Tính lại `set` từ đầu** cho mỗi `j`. Nhưng `s[i..j+1]` chỉ hơn `s[i..j]` đúng **một ký tự** — ta chỉ cần `seen.add(s[j])`.
2. **Không biết dừng.** Hãy để ý một tính chất rất quan trọng:

> **Thêm ký tự vào cuối chỉ làm số loại ký tự giữ nguyên hoặc tăng, không bao giờ giảm.**

Nên khi `s[i..j]` đã có **3 loại**, mọi `s[i..j']` với `j' > j` cũng có **≥ 3 loại** ⇒ không cần xét tiếp ⇒ `break`.

```text
với mỗi i:
    seen = set()
    với j từ i sang phải:
        seen.add(s[j])
        nếu len(seen) > 2: break        # dừng sớm
        best = max(best, j - i + 1)
```

### Trace Cách 1b với `"eceba"`

Mỗi dòng là một vị trí bắt đầu `i`: ta liệt kê các đoạn mở rộng được, rồi ký tự làm "vỡ" điều kiện (nếu có).

In [9]:
def longest_two_distinct_quadratic(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(s[j])
            if len(seen) > 2:          # đã có ký tự thứ 3 -> mọi j lớn hơn cũng hỏng
                break
            best = max(best, j - i + 1)
    return best


print(longest_two_distinct_quadratic("eceba"))     # 3
print(longest_two_distinct_quadratic("ccaabbb"))   # 5


def trace_quadratic(s):
    best = 0
    for i in range(len(s)):
        seen, grown = set(), []
        stop = "hết chuỗi"
        for j in range(i, len(s)):
            seen.add(s[j])
            if len(seen) > 2:
                stop = f"gặp {s[j]!r} tại j={j} -> 3 loại, break"
                break
            grown.append(s[i:j + 1])
            best = max(best, j - i + 1)
        print(f"i={i}: {grown} | {stop} | best={best}")
    return best

trace_quadratic("eceba")

3
5
i=0: ['e', 'ec', 'ece'] | gặp 'b' tại j=3 -> 3 loại, break | best=3
i=1: ['c', 'ce'] | gặp 'b' tại j=3 -> 3 loại, break | best=3
i=2: ['e', 'eb'] | gặp 'a' tại j=4 -> 3 loại, break | best=3
i=3: ['b', 'ba'] | hết chuỗi | best=3
i=4: ['a'] | hết chuỗi | best=3


3

### Cách 1b còn vướng ở đâu?

Nhìn dòng `i=0` và `i=1` ở trên: với `i=0` ta đã đi qua `"ece"`; sang `i=1` ta **vứt hết** `seen` và đi lại `"c"`, `"ce"` — những ký tự vừa đọc xong. Công việc bị **lặp lại**.

Trường hợp tệ nhất là khi chuỗi có ít loại ký tự, ví dụ `s = "aaaa...a"` (toàn chữ `a`): **không bao giờ `break`**, mỗi `i` lại chạy tới cuối chuỗi ⇒ `n(n+1)/2` bước. Với `n = 10^5` là khoảng **5 tỉ** bước — quá chậm cho Python.

Hãy đếm số bước thật:

In [10]:
def count_steps_quadratic(s):
    steps = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            steps += 1
            seen.add(s[j])
            if len(seen) > 2:
                break
    return steps

for n in [10, 100, 1000]:
    print(f"n={n:>5}, s='a'*n -> {count_steps_quadratic('a' * n):>8,} bước  (n(n+1)/2 = {n * (n + 1) // 2:,})")

n=   10, s='a'*n ->       55 bước  (n(n+1)/2 = 55)
n=  100, s='a'*n ->    5,050 bước  (n(n+1)/2 = 5,050)
n= 1000, s='a'*n ->  500,500 bước  (n(n+1)/2 = 500,500)


## 6. Bước đột phá: `left` tốt nhất chỉ đi tới, không bao giờ lùi

Đổi góc nhìn: thay vì cố định **đầu trái** `i`, hãy cố định **đầu phải** `right` và hỏi:

> Trong các đoạn **kết thúc tại `right`** và hợp lệ, đoạn nào dài nhất?

Đoạn dài nhất kết thúc tại `right` là `s[L(right)..right]`, với `L(right)` là **vị trí trái nhỏ nhất** còn hợp lệ.

Hai sự thật đơn giản (cùng gốc "thêm ký tự không làm giảm số loại"):

1. **Đoạn con của đoạn hợp lệ cũng hợp lệ.** Nếu `s[l..r]` có ≤ 2 loại thì mọi đoạn nằm trong nó cũng ≤ 2 loại.
2. Từ (1): `s[L(right)..right]` hợp lệ ⇒ `s[L(right)..right-1]` hợp lệ ⇒ `L(right-1) <= L(right)`.

Tức là dãy `L(0), L(1), L(2), ...` **không giảm** (monotonic — đơn điệu). Khi `right` tăng, `left` tốt nhất chỉ **đứng yên hoặc tiến lên**. Đây là "giấy phép" để dùng **hai con trỏ cùng chiều** — sliding window.

Kiểm chứng bằng code (tính `L(right)` bằng brute force):

In [11]:
def smallest_valid_left(s):
    result = []
    for right in range(len(s)):
        L = right
        while L - 1 >= 0 and len(set(s[L - 1:right + 1])) <= 2:   # lùi trái khi còn hợp lệ
            L -= 1
        result.append(L)
    return result

for s in ["eceba", "ccaabbb", "abaccc"]:
    Ls = smallest_valid_left(s)
    lengths = [r - L + 1 for r, L in enumerate(Ls)]
    print(f"{s:8} L(right) = {Ls} | độ dài = {lengths} | max = {max(lengths)}")

eceba    L(right) = [0, 0, 0, 2, 3] | độ dài = [1, 2, 3, 2, 2] | max = 3
ccaabbb  L(right) = [0, 0, 0, 0, 2, 2, 2] | độ dài = [1, 2, 3, 4, 3, 4, 5] | max = 5
abaccc   L(right) = [0, 0, 0, 2, 2, 2] | độ dài = [1, 2, 3, 2, 3, 4] | max = 4


Nhìn `"ccaabbb"`: `L = [0, 0, 0, 0, 2, 2, 2]` — không bao giờ giảm. Khi `right = 4` (chữ `b` đầu tiên), `left` phải nhảy từ `0` lên `2` để bỏ hết chữ `c`; sau đó nó đứng yên.

Đáp án = `max(right - L(right) + 1)` = `5`. Vậy nhiệm vụ của sliding window là: **duy trì `left = L(right)` một cách rẻ tiền** khi `right` tiến dần từ trái sang phải. Ba cách tiếp theo khác nhau ở chỗ "duy trì" bằng cách nào.

## 7. Cách 2 (cách chính) — Sliding window + `dict` đếm tần suất

Giữ `count` = bảng đếm các ký tự **trong cửa sổ** `s[left..right]`. Khi đó `len(count)` = số loại ký tự trong cửa sổ.

Mỗi vòng lặp với `right = 0, 1, ..., n-1`:

1. **Mở rộng:** thêm `ch = s[right]` vào `count`.
2. **Co cho tới khi hợp lệ:** `while len(count) > 2`: bớt `s[left]` khỏi `count` (xóa key nếu về 0), `left += 1`.
3. **Ghi nhận:** lúc này cửa sổ hợp lệ, `best = max(best, right - left + 1)`.

Thứ tự **thêm → co → ghi nhận** là bắt buộc: ghi nhận trước khi co sẽ tính cả cửa sổ chưa hợp lệ (xem mục lỗi thường gặp).

`enumerate(s)` trả về từng cặp `(chỉ số, ký tự)`: `(0, 'e'), (1, 'c'), ...` — tiện vì ta cần cả `right` lẫn `ch`. Chạy thử các ví dụ:

In [12]:
def longest_two_distinct_sliding(s: str) -> int:
    count = {}          # ký tự -> số lần xuất hiện trong s[left..right]
    left = 0
    best = 0
    for right, ch in enumerate(s):
        # 1) mở rộng
        count[ch] = count.get(ch, 0) + 1
        # 2) co cho tới khi còn tối đa 2 loại
        while len(count) > 2:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        # 3) ghi nhận cửa sổ hợp lệ
        best = max(best, right - left + 1)
    return best


for s in ["eceba", "ccaabbb", "a", "abababab", "abcdef", "abaccc"]:
    print(f"{s!r:12} -> {longest_two_distinct_sliding(s)}")

'eceba'      -> 3
'ccaabbb'    -> 5
'a'          -> 1
'abababab'   -> 8
'abcdef'     -> 2
'abaccc'     -> 4


## 8. Mô phỏng Cách 2

### 8.1. Trace `s = "eceba"`

| `right` | `ch` | `count` sau khi thêm | Co? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0 | e | {e:1} | không | 0 | `e` | 1 |
| 1 | c | {e:1, c:1} | không | 0 | `ec` | 2 |
| 2 | e | {e:2, c:1} | không | 0 | `ece` | 3 |
| 3 | b | {e:2, c:1, b:1} → 3 loại | bỏ `e`@0 (e:1), bỏ `c`@1 (xóa c) | 2 | `eb` | 3 |
| 4 | a | {e:1, b:1, a:1} → 3 loại | bỏ `e`@2 (xóa e) | 3 | `ba` | 3 |

Dòng `right = 3` là nhánh đáng chú ý: bỏ `e`@0 **chưa đủ** (vẫn còn một `e` ở vị trí 2), phải bỏ tiếp `c`@1 thì chữ `c` mới **biến mất hẳn** khỏi cửa sổ. Đó là lý do cần `while`.

In [13]:
def trace_sliding(s):
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        print(f"right={right} ch={ch!r}: thêm -> {count}" + ("  (3 loại!)" if len(count) > 2 else ""))
        while len(count) > 2:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
            print(f"    co: bỏ s[{left - 1}]={out!r} -> {count}, left={left}")
        best = max(best, right - left + 1)
        print(f"    cửa sổ [{left}..{right}] = {s[left:right + 1]!r}, dài {right - left + 1}, best={best}")
    return best

print("Kết quả:", trace_sliding("eceba"))

right=0 ch='e': thêm -> {'e': 1}
    cửa sổ [0..0] = 'e', dài 1, best=1
right=1 ch='c': thêm -> {'e': 1, 'c': 1}
    cửa sổ [0..1] = 'ec', dài 2, best=2
right=2 ch='e': thêm -> {'e': 2, 'c': 1}
    cửa sổ [0..2] = 'ece', dài 3, best=3
right=3 ch='b': thêm -> {'e': 2, 'c': 1, 'b': 1}  (3 loại!)
    co: bỏ s[0]='e' -> {'e': 1, 'c': 1, 'b': 1}, left=1
    co: bỏ s[1]='c' -> {'e': 1, 'b': 1}, left=2
    cửa sổ [2..3] = 'eb', dài 2, best=3
right=4 ch='a': thêm -> {'e': 1, 'b': 1, 'a': 1}  (3 loại!)
    co: bỏ s[2]='e' -> {'b': 1, 'a': 1}, left=3
    cửa sổ [3..4] = 'ba', dài 2, best=3
Kết quả: 3


### 8.2. Trace `s = "ccaabbb"`

| `right` | `ch` | `count` sau khi thêm | Co? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0 | c | {c:1} | không | 0 | `c` | 1 |
| 1 | c | {c:2} | không | 0 | `cc` | 2 |
| 2 | a | {c:2, a:1} | không | 0 | `cca` | 3 |
| 3 | a | {c:2, a:2} | không | 0 | `ccaa` | 4 |
| 4 | b | {c:2, a:2, b:1} → 3 loại | bỏ `c`@0, bỏ `c`@1 (xóa c) | 2 | `aab` | 4 |
| 5 | b | {a:2, b:2} | không | 2 | `aabb` | 4 |
| 6 | b | {a:2, b:3} | không | 2 | `aabbb` | **5** |

Để ý: tại `right = 4`, cửa sổ **ngắn đi** (từ 4 xuống 3), `best` vẫn giữ 4. Hai bước sau cửa sổ dài lại và vượt kỷ lục.

In [14]:
print("Kết quả:", trace_sliding("ccaabbb"))

right=0 ch='c': thêm -> {'c': 1}
    cửa sổ [0..0] = 'c', dài 1, best=1
right=1 ch='c': thêm -> {'c': 2}
    cửa sổ [0..1] = 'cc', dài 2, best=2
right=2 ch='a': thêm -> {'c': 2, 'a': 1}
    cửa sổ [0..2] = 'cca', dài 3, best=3
right=3 ch='a': thêm -> {'c': 2, 'a': 2}
    cửa sổ [0..3] = 'ccaa', dài 4, best=4
right=4 ch='b': thêm -> {'c': 2, 'a': 2, 'b': 1}  (3 loại!)
    co: bỏ s[0]='c' -> {'c': 1, 'a': 2, 'b': 1}, left=1
    co: bỏ s[1]='c' -> {'a': 2, 'b': 1}, left=2
    cửa sổ [2..4] = 'aab', dài 3, best=4
right=5 ch='b': thêm -> {'a': 2, 'b': 2}
    cửa sổ [2..5] = 'aabb', dài 4, best=4
right=6 ch='b': thêm -> {'a': 2, 'b': 3}
    cửa sổ [2..6] = 'aabbb', dài 5, best=5
Kết quả: 5


### 8.3. Vì sao Cách 2 là `O(n)` dù có `while` lồng trong `for`?

Nhìn hình thức "vòng `while` nằm trong vòng `for`" rất dễ tưởng là `O(n²)`. Nhưng hãy đếm theo **từng ký tự**:

- Mỗi vị trí được **thêm** vào cửa sổ đúng **1 lần** (khi `right` đi qua nó).
- Mỗi vị trí bị **bỏ** khỏi cửa sổ **nhiều nhất 1 lần** (khi `left` đi qua nó), vì `left` chỉ tăng và không vượt quá `n`.

Tổng số lần chạy thân `while` trong **cả** thuật toán ≤ `n`. Cộng lại: ≤ `2n` thao tác ⇒ **`O(n)`**. Kiểu lập luận "tính tổng cả quá trình thay vì từng vòng" gọi là **phân tích khấu hao** (amortized analysis).

**Space:** `count` không bao giờ có quá 3 key ⇒ **`O(1)`**.

Kiểm chứng bằng cách đếm số thao tác thêm/bỏ, so với Cách 1b trên cùng chuỗi toàn `a`:

In [15]:
def count_ops_sliding(s):
    count, left, adds, removes = {}, 0, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        adds += 1
        while len(count) > 2:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
            removes += 1
    return adds, removes

import random
random.seed(159)
tests = {"'a'*1000": "a" * 1000,
         "'abc'*333": "abc" * 333,
         "ngẫu nhiên 1000 (abc)": "".join(random.choice("abc") for _ in range(1000))}
for name, s in tests.items():
    adds, removes = count_ops_sliding(s)
    print(f"{name:24}: thêm={adds}, bỏ={removes}, tổng={adds + removes:>5} (<= 2n={2 * len(s)}) | Cách 1b: {count_steps_quadratic(s):>7,} bước")

'a'*1000                : thêm=1000, bỏ=0, tổng= 1000 (<= 2n=2000) | Cách 1b: 500,500 bước
'abc'*333               : thêm=999, bỏ=997, tổng= 1996 (<= 2n=1998) | Cách 1b:   2,994 bước
ngẫu nhiên 1000 (abc)   : thêm=1000, bỏ=997, tổng= 1997 (<= 2n=2000) | Cách 1b:   5,531 bước


## 9. Cách 3 — Lưu vị trí xuất hiện **cuối**, `left` nhảy một phát

### Cách 2 còn gì để cải tiến?

Ở `"ccaabbb"`, `right = 4`, Cách 2 phải bỏ `c` **từng cái một** (`c`@0 rồi `c`@1). Nếu cửa sổ là `"cccccccab"` thì phải bỏ 7 lần. Tổng vẫn `O(n)`, nhưng ta có thể làm gọn hơn: **tính thẳng** `left` mới.

### Ý tưởng

Thay vì đếm, lưu `last[ch]` = **vị trí xuất hiện cuối cùng** của `ch` trong cửa sổ.

Khi thêm `s[right]` làm cửa sổ có **3** loại ký tự, ta phải loại bỏ hẳn **một** loại. Loại nào? Loại có **`last` nhỏ nhất** — gọi là `victim`:

- Để `victim` biến mất khỏi cửa sổ, `left` phải vượt qua **lần xuất hiện cuối** của nó ⇒ `left = last[victim] + 1`.
- Hai loại còn lại đều có lần xuất hiện cuối **sau** `last[victim]`, nên vẫn còn trong cửa sổ — và ta giữ được cửa sổ **dài nhất** có thể.
- Nếu chọn loại khác (có `last` lớn hơn) thì `left` phải nhảy **xa hơn** ⇒ cửa sổ bị cắt cụt không cần thiết (mất luôn cả loại có `last` nhỏ hơn), và `last` còn giữ một key "ma" của ký tự đã nằm ngoài cửa sổ ⇒ `len(last)` sai từ đó về sau.

Ví dụ `"ccaabbb"`, `right = 4`: `last = {c:1, a:3, b:4}` ⇒ `victim = 'c'`, `left = 1 + 1 = 2` ⇒ cửa sổ `"aab"`. Đúng bằng kết quả Cách 2, nhưng chỉ tốn **một** phép gán.

In [16]:
s = "ccaabbb"
last = {}
for right in range(5):            # xử lý tới right = 4
    last[s[right]] = right        # ghi đè: luôn giữ vị trí MỚI NHẤT
print("last =", last)
victim = min(last, key=last.get)
print("victim =", victim, "| left mới =", last[victim] + 1)
del last[victim]
print("sau khi xóa:", last, "| cửa sổ s[2..4] =", repr(s[2:5]))

last = {'c': 1, 'a': 3, 'b': 4}
victim = c | left mới = 2
sau khi xóa: {'a': 3, 'b': 4} | cửa sổ s[2..4] = 'aab'


Ghép thành hàm hoàn chỉnh. Chú ý `last[ch] = right` luôn **ghi đè**, nên dict tự động giữ vị trí mới nhất; và `len(last)` = số loại ký tự trong cửa sổ (vì ta `del` loại bị loại).

In [17]:
def longest_two_distinct_last_index(s: str) -> int:
    last = {}           # ký tự -> vị trí xuất hiện cuối cùng trong cửa sổ
    left = 0
    best = 0
    for right, ch in enumerate(s):
        last[ch] = right
        if len(last) > 2:                       # 3 loại: loại bỏ 1 loại
            victim = min(last, key=last.get)    # loại có lần xuất hiện cuối sớm nhất
            left = last[victim] + 1             # nhảy thẳng qua nó
            del last[victim]
        best = max(best, right - left + 1)
    return best


for s in ["eceba", "ccaabbb", "abaccc", "a", "abcdef"]:
    print(f"{s!r:10} -> {longest_two_distinct_last_index(s)}")

'eceba'    -> 3
'ccaabbb'  -> 5
'abaccc'   -> 4
'a'        -> 1
'abcdef'   -> 2


Vì sao chỉ cần `if` (không cần `while`) ở đây? Mỗi bước chỉ thêm **một** ký tự ⇒ số loại tăng nhiều nhất từ 2 lên 3 ⇒ chỉ cần loại **một** loại là hợp lệ lại ngay.

### Trace Cách 3 với `"eceba"`

| `right` | `ch` | `last` sau khi gán | 3 loại? | `victim` | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|---|
| 0 | e | {e:0} | không | | 0 | `e` | 1 |
| 1 | c | {e:0, c:1} | không | | 0 | `ec` | 2 |
| 2 | e | {e:2, c:1} | không | | 0 | `ece` | 3 |
| 3 | b | {e:2, c:1, b:3} | có | `c` (last=1) | 2 | `eb` | 3 |
| 4 | a | {e:2, b:3, a:4} | có | `e` (last=2) | 3 | `ba` | 3 |

Dòng `right = 3` cho thấy điểm hay: chữ `e` xuất hiện **đầu tiên** trong cửa sổ, nhưng `victim` lại là `c` — vì `e` còn một bản ở vị trí 2, mới hơn `c`@1. Nếu ngây thơ "bỏ ký tự ở `s[left]`" thì sẽ sai (xem mục lỗi).

In [18]:
def trace_last_index(s):
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        last[ch] = right
        snapshot = dict(last)                   # chụp lại last ngay sau khi gán
        note = ""
        if len(last) > 2:
            victim = min(last, key=last.get)
            note = f" -> victim={victim!r}, left={last[victim] + 1}"
            left = last[victim] + 1
            del last[victim]
        best = max(best, right - left + 1)
        print(f"right={right} {ch!r} last={snapshot}{note} | cửa sổ {s[left:right + 1]!r} | best={best}")
    return best

print("Kết quả:", trace_last_index("eceba"))
print()
print("Kết quả:", trace_last_index("ccaabbb"))

right=0 'e' last={'e': 0} | cửa sổ 'e' | best=1
right=1 'c' last={'e': 0, 'c': 1} | cửa sổ 'ec' | best=2
right=2 'e' last={'e': 2, 'c': 1} | cửa sổ 'ece' | best=3
right=3 'b' last={'e': 2, 'c': 1, 'b': 3} -> victim='c', left=2 | cửa sổ 'eb' | best=3
right=4 'a' last={'e': 2, 'b': 3, 'a': 4} -> victim='e', left=3 | cửa sổ 'ba' | best=3
Kết quả: 3

right=0 'c' last={'c': 0} | cửa sổ 'c' | best=1
right=1 'c' last={'c': 1} | cửa sổ 'cc' | best=2
right=2 'a' last={'c': 1, 'a': 2} | cửa sổ 'cca' | best=3
right=3 'a' last={'c': 1, 'a': 3} | cửa sổ 'ccaa' | best=4
right=4 'b' last={'c': 1, 'a': 3, 'b': 4} -> victim='c', left=2 | cửa sổ 'aab' | best=4
right=5 'b' last={'a': 3, 'b': 5} | cửa sổ 'aabb' | best=4
right=6 'b' last={'a': 3, 'b': 6} | cửa sổ 'aabbb' | best=5
Kết quả: 5


### Độ phức tạp Cách 3

- Mỗi bước: gán `last[ch]` `O(1)`; khi có 3 loại thì `min` trên **3** key ⇒ `O(1)`.
- **Time `O(n)`**, **Space `O(1)`** (`last` tối đa 3 key).

Khi tổng quát lên **k** loại ký tự (bài 340), `min` duyệt `k+1` key ⇒ `O(n·k)`. Còn Cách 2 tổng quát vẫn `O(n)`. Vậy Cách 3 gọn với `k = 2`, nhưng Cách 2 "tổng quát hóa tốt" hơn. (Có thể đưa về `O(n)` cho Cách 3 bằng `OrderedDict` — để dành cho bạn tìm hiểu thêm.)

## 10. Cách 4 — Cửa sổ "không co" (non-shrinking window)

### Câu hỏi gợi mở

Đề chỉ hỏi **độ dài lớn nhất**. Khi đã tìm được một cửa sổ hợp lệ dài 4, ta **không còn quan tâm** đến cửa sổ ngắn hơn 4 nữa. Vậy tại sao phải co cửa sổ xuống 3, 2...?

### Ý tưởng

Cho cửa sổ **không bao giờ ngắn đi**:

1. Mỗi bước `right += 1` (thêm `s[right]`) — cửa sổ dài thêm 1.
2. Nếu cửa sổ **không hợp lệ** (`len(count) > 2`), chỉ dịch `left` **đúng 1 bước** bằng `if` — cửa sổ **giữ nguyên** độ dài, như một khung cố định trượt sang phải.
3. Không cần biến `best`: **độ dài cửa sổ lúc kết thúc** chính là đáp án, tức `n - left`.

Hệ quả lạ: cửa sổ có lúc **không hợp lệ** (chứa 3, thậm chí 4 loại ký tự)! Nhưng **độ dài** của nó luôn đúng bằng kỷ lục.

In [19]:
def longest_two_distinct_nonshrinking(s: str) -> int:
    count = {}
    left = 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        if len(count) > 2:                  # IF, không phải WHILE: chỉ trượt 1 bước
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
    return len(s) - left                     # độ dài cửa sổ cuối = kỷ lục


for s in ["eceba", "ccaabbb", "abaccc", "a", "abcdef", "aabbcdef"]:
    print(f"{s!r:11} -> {longest_two_distinct_nonshrinking(s)}")

'eceba'     -> 3
'ccaabbb'   -> 5
'abaccc'    -> 4
'a'         -> 1
'abcdef'    -> 2
'aabbcdef'  -> 4


### Trace Cách 4 với `"ccaabbb"`

| `right` | `ch` | `count` sau khi thêm | > 2 loại? | Hành động | `left` | Cửa sổ | Hợp lệ? | Độ dài |
|---|---|---|---|---|---|---|---|---|
| 0 | c | {c:1} | không | — | 0 | `c` | có | 1 |
| 1 | c | {c:2} | không | — | 0 | `cc` | có | 2 |
| 2 | a | {c:2, a:1} | không | — | 0 | `cca` | có | 3 |
| 3 | a | {c:2, a:2} | không | — | 0 | `ccaa` | có | 4 |
| 4 | b | {c:2, a:2, b:1} | **có** | bỏ `c`@0 | 1 | `caab` | **không** | 4 |
| 5 | b | {c:1, a:2, b:2} | **có** | bỏ `c`@1 (xóa c) | 2 | `aabb` | có | 4 |
| 6 | b | {a:2, b:3} | không | — | 2 | `aabbb` | có | **5** |

Kết quả `n - left = 7 - 2 = 5`. Ở `right = 4` cửa sổ `"caab"` **không hợp lệ** nhưng ta vẫn để nó dài 4 — vì 4 là kỷ lục đã đạt được trước đó (`"ccaa"`).

In [20]:
def trace_nonshrinking(s):
    count, left = {}, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        action = "—"
        if len(count) > 2:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
            action = f"bỏ s[{left - 1}]={out!r}"
        window = s[left:right + 1]
        ok = "hợp lệ" if len(set(window)) <= 2 else "KHÔNG hợp lệ"
        print(f"right={right} ch={ch!r} {action:12} left={left} cửa sổ={window!r:8} {ok:13} dài={len(window)}")
    print("Đáp án = n - left =", len(s) - left)

trace_nonshrinking("ccaabbb")

right=0 ch='c' —            left=0 cửa sổ='c'      hợp lệ        dài=1
right=1 ch='c' —            left=0 cửa sổ='cc'     hợp lệ        dài=2
right=2 ch='a' —            left=0 cửa sổ='cca'    hợp lệ        dài=3
right=3 ch='a' —            left=0 cửa sổ='ccaa'   hợp lệ        dài=4
right=4 ch='b' bỏ s[0]='c'  left=1 cửa sổ='caab'   KHÔNG hợp lệ  dài=4
right=5 ch='b' bỏ s[1]='c'  left=2 cửa sổ='aabb'   hợp lệ        dài=4
right=6 ch='b' —            left=2 cửa sổ='aabbb'  hợp lệ        dài=5
Đáp án = n - left = 5


Một ví dụ "cực đoan" hơn: `"aabbcdef"`. Sau khi đạt kỷ lục `"aabb"` (dài 4), cửa sổ trượt nguyên khối và chứa tới **4 loại** ký tự — vậy mà đáp án vẫn đúng là 4.

In [21]:
trace_nonshrinking("aabbcdef")

right=0 ch='a' —            left=0 cửa sổ='a'      hợp lệ        dài=1
right=1 ch='a' —            left=0 cửa sổ='aa'     hợp lệ        dài=2
right=2 ch='b' —            left=0 cửa sổ='aab'    hợp lệ        dài=3
right=3 ch='b' —            left=0 cửa sổ='aabb'   hợp lệ        dài=4
right=4 ch='c' bỏ s[0]='a'  left=1 cửa sổ='abbc'   KHÔNG hợp lệ  dài=4
right=5 ch='d' bỏ s[1]='a'  left=2 cửa sổ='bbcd'   KHÔNG hợp lệ  dài=4
right=6 ch='e' bỏ s[2]='b'  left=3 cửa sổ='bcde'   KHÔNG hợp lệ  dài=4
right=7 ch='f' bỏ s[3]='b'  left=4 cửa sổ='cdef'   KHÔNG hợp lệ  dài=4
Đáp án = n - left = 4


### Vì sao Cách 4 vẫn đúng? (chứng minh ngắn)

Gọi `best(r)` = độ dài chuỗi con hợp lệ dài nhất nằm trong `s[0..r]`. Ta chứng minh **bất biến**:

> Sau khi xử lý xong `right = r`, độ dài cửa sổ `r - left + 1` đúng bằng `best(r)`.

- **Lúc đầu** (`r = 0`): cửa sổ `s[0..0]` dài 1, hợp lệ ⇒ đúng.
- **Bước `r-1 → r`**: theo bất biến, trước khi thêm, cửa sổ dài `best(r-1)`. Thêm `s[r]` ⇒ cửa sổ `W = s[left..r]` dài `best(r-1) + 1`.
  - Nếu `W` **hợp lệ**: ta đã có một chuỗi con hợp lệ dài `best(r-1) + 1`. Không thể dài hơn nữa: một đoạn hợp lệ dài `best(r-1) + 2` kết thúc tại `r` thì bỏ ký tự cuối sẽ cho đoạn hợp lệ dài `best(r-1) + 1` nằm trong `s[0..r-1]` — mâu thuẫn. Vậy `best(r) = best(r-1) + 1` = độ dài cửa sổ. Ta không dịch `left`. ✔
  - Nếu `W` **không hợp lệ**: `W` chính là **đoạn duy nhất** dài `best(r-1) + 1` kết thúc tại `r`, mà nó hỏng ⇒ không có đoạn hợp lệ nào dài như thế kết thúc tại `r` (đoạn dài hơn càng không) ⇒ `best(r) = best(r-1)`. Ta dịch `left` 1 bước ⇒ cửa sổ dài `best(r-1)`. ✔

Kết thúc: `n - left = best(n-1)` = đáp án.

Điểm then chốt: điều kiện "`len(count) > 2`" vẫn được tính **đúng trên cửa sổ hiện tại** (dù cửa sổ có thể hỏng), vì ta vẫn cập nhật `count` mỗi lần thêm/bỏ. Bất biến của Cách 4 là về **độ dài**, không phải về **tính hợp lệ**.

### Hệ quả và giới hạn

- Mã ngắn hơn, không cần `best`, mỗi bước làm ≤ 1 thao tác bỏ ⇒ **Time `O(n)`**.
- `count` có thể chứa **nhiều hơn 3** key (thấy ở `"aabbcdef"`), nhưng không vượt số loại chữ cái (52) ⇒ **Space `O(1)`** (chính xác hơn `O(min(n, Σ))`, Σ = kích thước bảng chữ cái).
- **Chỉ dùng được khi đề hỏi độ dài lớn nhất.** Nếu đề hỏi **chính chuỗi con**, hay **đếm số chuỗi con hợp lệ**, cửa sổ cuối có thể là rác (như `"cdef"` ở trên) ⇒ phải dùng Cách 2.

## 11. Code nộp LeetCode

Bản nộp chính dùng **Cách 2** — dễ đọc, dễ tổng quát hóa, đúng tinh thần "cửa sổ luôn hợp lệ". Bản Cách 3 và Cách 4 để ở ô kế tiếp (đổi tên class để không đè nhau trong notebook; khi nộp, đổi lại thành `Solution`).

### Bộ test

1. Hai ví dụ của đề.
2. Chuỗi 1 ký tự, 2 ký tự.
3. Toàn một loại ký tự; chỉ 2 loại xen kẽ ⇒ đáp án là cả chuỗi.
4. Mọi ký tự khác nhau ⇒ `2`.
5. Đáp án nằm ở **cuối** chuỗi, ở **đầu** chuỗi.
6. Chữ hoa và chữ thường là **khác nhau** (`"aA"` có 2 loại).
7. Ca "bẫy" cho Cách 3: ký tự ở `s[left]` không phải `victim` (`"abaccc"`).

In [22]:
class Solution:
    def lengthOfLongestSubstringTwoDistinct(self, s: str) -> int:
        count = {}
        left = 0
        best = 0
        for right, ch in enumerate(s):
            count[ch] = count.get(ch, 0) + 1
            while len(count) > 2:
                out = s[left]
                count[out] -= 1
                if count[out] == 0:
                    del count[out]
                left += 1
            best = max(best, right - left + 1)
        return best


class SolutionLastIndex:
    def lengthOfLongestSubstringTwoDistinct(self, s: str) -> int:
        last, left, best = {}, 0, 0
        for right, ch in enumerate(s):
            last[ch] = right
            if len(last) > 2:
                victim = min(last, key=last.get)
                left = last[victim] + 1
                del last[victim]
            best = max(best, right - left + 1)
        return best


class SolutionNonShrinking:
    def lengthOfLongestSubstringTwoDistinct(self, s: str) -> int:
        count, left = {}, 0
        for ch in s:
            count[ch] = count.get(ch, 0) + 1
            if len(count) > 2:
                out = s[left]
                count[out] -= 1
                if count[out] == 0:
                    del count[out]
                left += 1
        return len(s) - left

In [23]:
def check(name, s, expected):
    results = [cls().lengthOfLongestSubstringTwoDistinct(s)
               for cls in (Solution, SolutionLastIndex, SolutionNonShrinking)]
    assert results == [expected] * 3, (name, s, results, expected)
    print(f"{name:<28} {s!r:14} -> {expected}  PASS")


check("example 1", "eceba", 3)
check("example 2", "ccaabbb", 5)
check("1 ký tự", "a", 1)
check("2 ký tự khác nhau", "ab", 2)
check("toàn một loại", "zzzzzz", 6)
check("2 loại xen kẽ", "abababab", 8)
check("mọi ký tự khác nhau", "abcdef", 2)
check("đáp án ở cuối", "abcbbbbcc", 8)
check("đáp án ở đầu", "aabbbcd", 5)
check("hoa/thường khác nhau", "aAaAb", 4)
check("bẫy s[left] != victim", "abaccc", 4)
print("Tất cả test đều PASS!")

example 1                    'eceba'        -> 3  PASS
example 2                    'ccaabbb'      -> 5  PASS
1 ký tự                      'a'            -> 1  PASS
2 ký tự khác nhau            'ab'           -> 2  PASS
toàn một loại                'zzzzzz'       -> 6  PASS
2 loại xen kẽ                'abababab'     -> 8  PASS
mọi ký tự khác nhau          'abcdef'       -> 2  PASS
đáp án ở cuối                'abcbbbbcc'    -> 8  PASS
đáp án ở đầu                 'aabbbcd'      -> 5  PASS
hoa/thường khác nhau         'aAaAb'        -> 4  PASS
bẫy s[left] != victim        'abaccc'       -> 4  PASS
Tất cả test đều PASS!


### Test đối chiếu chéo trên input ngẫu nhiên

Sinh chuỗi ngắn từ bảng chữ cái nhỏ (`"ab"`, `"abc"`, `"abcd"`) để có nhiều tình huống: lặp, xen kẽ, có ký tự thứ 3, thứ 4... So **6 hàm** với nhau, lấy brute force `O(n³)` làm trọng tài. `random.seed(159)` cố định để lần chạy nào cũng cùng dữ liệu.

### Chạy với `n = 10^5` (giới hạn của đề)

Các cách `O(n)` xử lý 100 nghìn ký tự trong chớp mắt. (Thời gian in ra phụ thuộc máy, chỉ để cảm nhận.) Không chạy Cách 1a/1b ở đây — với chuỗi toàn `a`, Cách 1b sẽ cần khoảng 5 tỉ bước.

In [24]:
random.seed(159)
for _ in range(2000):
    alphabet = random.choice(["ab", "abc", "abcd", "aAbB"])
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 14)))
    expected = longest_two_distinct_bruteforce(s)
    got = [
        longest_two_distinct_quadratic(s),
        longest_two_distinct_sliding(s),
        longest_two_distinct_last_index(s),
        longest_two_distinct_nonshrinking(s),
        Solution().lengthOfLongestSubstringTwoDistinct(s),
    ]
    assert got == [expected] * 5, (s, expected, got)
print("2000 bộ ngẫu nhiên: cả 6 hàm cho cùng kết quả.")


import time

big_cases = {
    "'a' * 10^5": "a" * 10**5,
    "ngẫu nhiên 10^5 (abc)": "".join(random.choice("abc") for _ in range(10**5)),
    "ngẫu nhiên 10^5 (52 chữ)": "".join(random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(10**5)),
}
for name, s in big_cases.items():
    t0 = time.perf_counter()
    a = longest_two_distinct_sliding(s)
    b = longest_two_distinct_last_index(s)
    c = longest_two_distinct_nonshrinking(s)
    dt = time.perf_counter() - t0
    assert a == b == c
    print(f"{name:26}: đáp án = {a:>6} | 3 cách O(n) chạy tổng cộng {dt:.2f} giây")

2000 bộ ngẫu nhiên: cả 6 hàm cho cùng kết quả.


'a' * 10^5                : đáp án = 100000 | 3 cách O(n) chạy tổng cộng 0.02 giây


ngẫu nhiên 10^5 (abc)     : đáp án =     31 | 3 cách O(n) chạy tổng cộng 0.05 giây


ngẫu nhiên 10^5 (52 chữ)  : đáp án =      6 | 3 cách O(n) chạy tổng cộng 0.07 giây


## 12. Vì sao Cách 2 đúng?

**Bất biến (invariant)** sau bước 2 (co) ở mỗi vòng `right`:

> `count` mô tả đúng các ký tự trong `s[left..right]`, cửa sổ có ≤ 2 loại, và `left` là **vị trí trái nhỏ nhất** sao cho `s[left..right]` hợp lệ (tức `left = L(right)` ở mục 6).

Lập luận:

1. `count` đúng: mỗi lần thêm `s[right]` ta tăng, mỗi lần `left` tiến ta giảm `s[left]` và xóa key về 0 ⇒ `len(count)` = số loại trong cửa sổ.
2. Hợp lệ: vòng `while` chỉ dừng khi `len(count) <= 2`.
3. `left` nhỏ nhất: theo mục 6, `L(right) >= L(right-1)`, và vòng trước ta có `left = L(right-1)`. Vòng `while` tăng `left` **từng bước một** và dừng **ngay lần đầu** cửa sổ hợp lệ ⇒ nó dừng đúng tại `L(right)`, không vượt quá.

Vì mọi chuỗi con hợp lệ đều **kết thúc ở một `right` nào đó**, và với mỗi `right` ta đã xét đoạn dài nhất kết thúc tại đó, nên `best = max(right - L(right) + 1)` là đáp án. Không bỏ sót, không tính nhầm đoạn không hợp lệ.

**Cách 3** đúng cùng lý do: `left = last[victim] + 1` đúng bằng `L(right)` — là vị trí nhỏ nhất làm `victim` biến mất, mà hai loại còn lại vẫn ở trong cửa sổ.

## 13. So sánh các cách

`n = len(s)`, `Σ` = số loại ký tự có thể có (≤ 52).

| Cách | Ý tưởng | Time | Space | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1a. Brute force | Mọi `(i, j)` + `len(set)` | `O(n³)` | `O(n)` | Hiển nhiên đúng, làm trọng tài | Quá chậm |
| 1b. Cố định trái + dừng sớm | Thêm dần vào `set`, `break` khi 3 loại | `O(n²)` | `O(1)` | Dùng tính "thêm không làm giảm" | Làm lại việc cũ mỗi `i` |
| **2. Sliding + đếm** | Thêm phải, `while` co trái | **`O(n)`** | `O(1)` | Chuẩn mực, tổng quát hóa `k` vẫn `O(n)` | Co từng bước một |
| 3. Sliding + vị trí cuối | `left = min(last) + 1` | `O(n)` (`O(n·k)` nếu tổng quát) | `O(1)` | `left` nhảy 1 phát, `if` là đủ | `min` tốn `O(k)` khi `k` lớn |
| 4. Không co | `if` dịch `left` 1 bước, đáp án `n - left` | `O(n)` | `O(min(n, Σ))` | Ngắn nhất, không cần `best` | Chỉ cho độ dài; cửa sổ có lúc không hợp lệ |

## 14. Lỗi thường gặp

### Lỗi 1 — Quên `del` khi đếm về 0 (hoặc dùng `Counter` mà không xóa)

Key có giá trị `0` vẫn được `len(count)` đếm ⇒ vòng `while` tưởng cửa sổ vẫn còn 3 loại và **co mãi**, `left` vượt cả `right` rồi chạy ra ngoài chuỗi.

In [25]:
def buggy_no_delete(s):
    count, left, best = Counter(), 0, 0
    for right, ch in enumerate(s):
        count[ch] += 1
        while len(count) > 2:          # key có value 0 vẫn được đếm!
            count[s[left]] -= 1        # thiếu: if count[...] == 0: del ...
            left += 1
        best = max(best, right - left + 1)
    return best

for s in ["aab", "eceba"]:
    try:
        print(s, "->", buggy_no_delete(s))
    except IndexError as e:
        print(s, "-> IndexError:", e, "(left chạy ra ngoài chuỗi)")

aab -> 3
eceba -> IndexError: string index out of range (left chạy ra ngoài chuỗi)


`"aab"` chạy "may mắn" vì chưa bao giờ có 3 loại; `"eceba"` thì vỡ ngay. **Sửa:** thêm `if count[out] == 0: del count[out]`.

### Lỗi 2 — Ghi nhận `best` trước khi co

Đặt `best = max(...)` **trước** vòng `while` ⇒ tính cả cửa sổ đang có 3 loại.

In [26]:
def buggy_record_before_shrink(s):
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        best = max(best, right - left + 1)     # SAI: cửa sổ có thể đang có 3 loại
        while len(count) > 2:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
    return best

print("bản lỗi :", buggy_record_before_shrink("eceba"))    # 4 ('eceb' bị tính)
print("bản đúng:", longest_two_distinct_sliding("eceba"))  # 3

bản lỗi : 4
bản đúng: 3


**Sửa:** thứ tự luôn là **thêm → co → ghi nhận**.

### Lỗi 3 — Cách 3 nhưng loại ký tự ở `s[left]` thay vì `victim`

Ký tự **đứng đầu** cửa sổ chưa chắc là ký tự **xuất hiện cuối sớm nhất**. Với `"abaccc"`: khi `right = 3` (chữ `c`), cửa sổ `"abac"`, `s[left] = 'a'` nhưng `a` còn ở vị trí 2; ký tự đúng phải bỏ là `b` (last = 1).

In [27]:
def buggy_drop_s_left(s):
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        last[ch] = right
        if len(last) > 2:
            victim = s[left]                 # SAI: nên là min(last, key=last.get)
            left = last[victim] + 1
            del last[victim]
        best = max(best, right - left + 1)
    return best

print("bản lỗi :", buggy_drop_s_left("abaccc"))             # 3 (bỏ mất 'a' ở vị trí 2)
print("bản đúng:", longest_two_distinct_last_index("abaccc"))  # 4 ('accc')

bản lỗi : 3
bản đúng: 4


**Sửa:** `victim = min(last, key=last.get)`.

### Lỗi 4 — Dùng Cách 4 để lấy **chính chuỗi con**

Cửa sổ cuối của Cách 4 có độ dài đúng nhưng **nội dung có thể không hợp lệ**.

In [28]:
def buggy_nonshrinking_substring(s):
    count, left = {}, 0
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
        if len(count) > 2:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
    return s[left:]                     # SAI nếu đề hỏi chuỗi con

sub = buggy_nonshrinking_substring("aabbcdef")
print(repr(sub), "dài", len(sub), "| số loại:", len(set(sub)))   # 'cdef' dài 4 nhưng 4 loại!

'cdef' dài 4 | số loại: 4


**Sửa:** nếu cần chuỗi con, dùng Cách 2 và lưu `(left, right)` mỗi khi `best` tăng.

### Lỗi 5 — Các lỗi "lặt vặt"

- **Quên `+ 1`** khi tính độ dài: `right - left` là sai, đúng là `right - left + 1`.
- **`min(last)` thiếu `key=`**: so sánh theo **bảng chữ cái** chứ không theo vị trí (đã thấy ở mục 3.5).
- **Dùng `set` thay `dict` trong sliding window**: khi bỏ `s[left]` khỏi `set`, ta không biết trong cửa sổ còn bản nào khác của ký tự đó không. `set` chỉ dùng được ở Cách 1b (làm lại từ đầu mỗi `i`).
- **Gộp chữ hoa/thường** (`s.lower()`): đề coi `'a'` và `'A'` là khác nhau.

## 15. Mẹo nhận dạng pattern

Khi đề có dạng:

- "**longest / shortest substring (subarray)** ... such that ..." — chuỗi con / mảng con **liên tục**;
- và điều kiện có tính **đơn điệu**: nếu đoạn hợp lệ thì đoạn con của nó cũng hợp lệ (hoặc ngược lại: nếu đoạn hỏng thì đoạn chứa nó cũng hỏng);

⇒ nghĩ ngay tới **sliding window** hai con trỏ cùng chiều, `O(n)`.

Các "điều kiện đơn điệu" hay gặp: *tối đa k loại ký tự*, *không có ký tự lặp*, *tổng ≤ target (số không âm)*, *số lần thay thế ≤ k*...

Khung chung (khắc vào đầu):

```python
left = 0
for right in range(n):
    thêm a[right] vào trạng thái
    while trạng thái vi phạm:
        bỏ a[left] khỏi trạng thái
        left += 1
    cập nhật đáp án với cửa sổ [left..right]
```

Nếu đề chỉ hỏi **độ dài dài nhất**, có thể dùng biến thể "không co": đổi `while` thành `if`, đáp án `n - left`.

## 16. Biến thể và bài liên quan

| Bài | Liên hệ với #159 |
|---|---|
| **340. Longest Substring with At Most K Distinct Characters** (Premium) | Thay số `2` bằng `k` — tổng quát trực tiếp. Có thể `k = 0` ⇒ đáp án 0. |
| **904. Fruit Into Baskets** | Y hệt #159 nhưng trên **mảng số nguyên** `fruits` (2 giỏ = 2 loại quả). |
| **3. Longest Substring Without Repeating Characters** | Điều kiện đổi thành "mỗi ký tự xuất hiện ≤ 1 lần". Cùng khung sliding window. |
| **992. Subarrays with K Different Integers** (Hard) | **Đếm** số mảng con có **đúng** K loại: `exactly(K) = atMost(K) - atMost(K-1)`, với `atMost` cộng dồn `right - left + 1`. |
| **424. Longest Repeating Character Replacement** | Sliding window + đếm tần suất; cũng có lời giải "không co" nổi tiếng. |
| **76. Minimum Window Substring** (Hard) | Sliding window tìm cửa sổ **ngắn nhất** thỏa mãn. |

Code dưới viết bản tổng quát cho **k** (bài 340) bằng Cách 2, rồi dùng lại nó cho #159 (`k = 2`) và #904.

In [29]:
def longest_k_distinct(seq, k: int) -> int:
    """Độ dài đoạn liên tục dài nhất có tối đa k loại phần tử (seq là str hoặc list)."""
    count, left, best = {}, 0, 0
    for right, x in enumerate(seq):
        count[x] = count.get(x, 0) + 1
        while len(count) > k:
            out = seq[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        best = max(best, right - left + 1)
    return best


# LeetCode 340 — ví dụ của đề
assert longest_k_distinct("eceba", 2) == 3
assert longest_k_distinct("aa", 1) == 2
assert longest_k_distinct("abc", 0) == 0
# LeetCode 159 là trường hợp k = 2
assert longest_k_distinct("ccaabbb", 2) == 5
# LeetCode 904 — Fruit Into Baskets (mảng số nguyên, k = 2)
assert longest_k_distinct([1, 2, 1], 2) == 3
assert longest_k_distinct([0, 1, 2, 2], 2) == 3
assert longest_k_distinct([1, 2, 3, 2, 2], 2) == 4

for k in range(4):
    print(f"k={k}: longest_k_distinct('eceebaaab', {k}) = {longest_k_distinct('eceebaaab', k)}")

k=0: longest_k_distinct('eceebaaab', 0) = 0
k=1: longest_k_distinct('eceebaaab', 1) = 3
k=2: longest_k_distinct('eceebaaab', 2) = 5
k=3: longest_k_distinct('eceebaaab', 3) = 7


## 17. Bài tập tự luyện

Làm theo thứ tự, mỗi bài khó hơn bài trước. **Đáp án ở cuối notebook** — hãy tự làm trước.

### Bài 1 — Đoán output (không chạy code)

`longest_two_distinct_sliding` trả về gì với từng chuỗi sau? Ghi ra cả **chuỗi con** đạt kỷ lục.

1. `"abaccc"`
2. `"aabbcc"`
3. `"abcabcabc"`
4. `"Aa"`
5. `"bbbbbab"`

Gợi ý mức 1: với mỗi chuỗi, thử tìm đoạn dài nhất chỉ gồm 2 chữ.
Gợi ý mức 2: `"aabbcc"` có hai đoạn cùng dài kỷ lục.
Gợi ý mức 3: `"Aa"` — nhớ rằng chữ hoa và chữ thường khác nhau.

### Bài 2 — Tính tay Cách 3

Lập bảng trace **Cách 3** (vị trí xuất hiện cuối) cho `s = "abaccc"` với các cột: `right`, `ch`, `last` sau khi gán, có 3 loại không, `victim`, `left`, cửa sổ, `best`.

Gợi ý mức 1: `last` chỉ có 3 key ở đúng **một** bước.
Gợi ý mức 2: ở bước đó, `last = {a: 2, b: 1, c: 3}`.
Gợi ý mức 3: `victim = 'b'` ⇒ `left = 2`.

### Bài 3 — Điền TODO: Cách 2 trả về **chính chuỗi con**

Sửa Cách 2 để trả về **chuỗi con dài nhất** (nếu nhiều chuỗi cùng dài, trả về chuỗi **xuất hiện trước**).

In [30]:
def longest_two_distinct_substring(s: str) -> str:
    count, left = {}, 0
    best_len, best_start = 0, 0
    for right, ch in enumerate(s):
        # TODO 1: thêm ch vào count
        # TODO 2: while còn hơn 2 loại: bỏ s[left], xóa key nếu về 0, left += 1
        # TODO 3: nếu cửa sổ dài hơn best_len (dùng >, không phải >=): cập nhật best_len, best_start
        pass
    return s[best_start:best_start + best_len]

# Tự làm xong rồi mới mở test:
# assert longest_two_distinct_substring("eceba") == "ece"
# assert longest_two_distinct_substring("ccaabbb") == "aabbb"
# assert longest_two_distinct_substring("aabbcc") == "aabb"

Gợi ý mức 1: khung giống hệt `longest_two_distinct_sliding`, chỉ thay dòng `best = max(...)`.
Gợi ý mức 2: dùng `>` để giữ chuỗi **xuất hiện trước** khi bằng độ dài.
Gợi ý mức 3: `if right - left + 1 > best_len: best_len, best_start = right - left + 1, left`.

### Bài 4 — Tự viết lại: Cách 4 (không co) cho **k** loại — LeetCode 340

Không nhìn mục 10, tự viết lời giải "cửa sổ không co" cho bài tổng quát `k` loại ký tự. Nhớ trường hợp `k = 0`.

In [31]:
def longest_k_distinct_nonshrinking(s: str, k: int) -> int:
    # TODO 1: count = {}, left = 0
    # TODO 2: for mỗi ký tự: thêm vào count
    # TODO 3: IF len(count) > k: bỏ s[left] (xóa key nếu về 0), left += 1
    # TODO 4: return len(s) - left
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert longest_k_distinct_nonshrinking("eceba", 2) == 3
# assert longest_k_distinct_nonshrinking("aa", 1) == 2
# assert longest_k_distinct_nonshrinking("abc", 0) == 0
# assert longest_k_distinct_nonshrinking("aabbcdef", 3) == 5

Gợi ý mức 1: chỉ cần đổi số `2` thành `k` trong `longest_two_distinct_nonshrinking`.
Gợi ý mức 2: với `k = 0`, mỗi bước đều có `1 > 0` loại ⇒ `left` tăng đều theo `right` ⇒ `len(s) - left = 0`. Không cần xử lý riêng!
Gợi ý mức 3: bất biến "độ dài cửa sổ = kỷ lục" vẫn đúng với mọi `k`, lập luận y hệt mục 10.

### Bài 5 — Biến thể: LeetCode 904 Fruit Into Baskets bằng Cách 3

Một hàng cây, `fruits[i]` là **loại quả** của cây thứ `i`. Bạn có **2 giỏ**, mỗi giỏ chỉ đựng **một loại** quả (không giới hạn số lượng). Bắt đầu từ một cây bất kỳ, đi sang phải, hái **đúng 1 quả** ở mỗi cây; dừng khi gặp loại quả không bỏ vào giỏ nào được. Hỏi hái được **nhiều quả nhất** bao nhiêu?

Ví dụ: `[1,2,1]` ⇒ 3; `[0,1,2,2]` ⇒ 3; `[1,2,3,2,2]` ⇒ 4.

Hãy viết lời giải bằng **Cách 3** (lưu vị trí xuất hiện cuối), signature theo LeetCode `totalFruit(self, fruits: List[int]) -> int`.

In [32]:
from typing import List


class FruitSolution:
    def totalFruit(self, fruits: List[int]) -> int:
        # TODO: last = {}; với mỗi right: last[fruits[right]] = right
        #       nếu len(last) > 2: victim = min(last, key=last.get); left = last[victim] + 1; del last[victim]
        #       cập nhật best
        pass

# Tự làm xong rồi mới mở test:
# assert FruitSolution().totalFruit([1, 2, 1]) == 3
# assert FruitSolution().totalFruit([0, 1, 2, 2]) == 3
# assert FruitSolution().totalFruit([1, 2, 3, 2, 2]) == 4
# assert FruitSolution().totalFruit([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]) == 5

Gợi ý mức 1: "2 giỏ, mỗi giỏ 1 loại" = "đoạn liên tục có tối đa 2 loại phần tử". Đây chính là #159 trên mảng số.
Gợi ý mức 2: code Cách 3 không dùng gì đặc thù của chuỗi — `fruits[right]` thay cho `s[right]` là xong.
Gợi ý mức 3: ví dụ cuối, đoạn tốt nhất là `[1, 2, 1, 1, 2]` (vị trí 3..7).

### Bài 6 — Mini challenge: **đếm** chuỗi con có **đúng** 2 loại ký tự

Viết `count_exactly_two(s)` trả về **số** chuỗi con (tính theo vị trí, trùng nội dung vẫn đếm riêng) có **đúng 2** loại ký tự. Yêu cầu `O(n)`.

Ví dụ: `"aab"` ⇒ các chuỗi con có đúng 2 loại là `"ab"`, `"aab"` ⇒ **2**. `"abc"` ⇒ `"ab"`, `"bc"` ⇒ **2**.

In [33]:
def count_at_most(s: str, k: int) -> int:
    # TODO: sliding window Cách 2 với giới hạn k; mỗi right cộng thêm (right - left + 1)
    pass


def count_exactly_two(s: str) -> int:
    # TODO: dùng count_at_most
    pass

# Tự làm xong rồi mới mở test:
# assert count_exactly_two("aab") == 2
# assert count_exactly_two("abc") == 2
# assert count_exactly_two("eceba") == 5

Gợi ý mức 1: với mỗi `right`, có bao nhiêu chuỗi con **hợp lệ kết thúc tại `right`**? Chính là các đoạn `[l..right]` với `left <= l <= right` (vì đoạn con của đoạn hợp lệ cũng hợp lệ) ⇒ `right - left + 1` đoạn.
Gợi ý mức 2: "đúng 2 loại" = "tối đa 2 loại" trừ "tối đa 1 loại".
Gợi ý mức 3: ở đây **bắt buộc** dùng `while` (Cách 2), không dùng được Cách 4 — vì ta cần `left` đúng bằng `L(right)` ở **mọi** bước, không chỉ độ dài.

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án bài 1

1. `"abaccc"` ⇒ **4**, chuỗi `"accc"` (`"aba"` chỉ dài 3).
2. `"aabbcc"` ⇒ **4**, `"aabb"` hoặc `"bbcc"`.
3. `"abcabcabc"` ⇒ **2**: mọi đoạn dài 3 đều có đủ `a, b, c`.
4. `"Aa"` ⇒ **2**: `'A'` và `'a'` là 2 loại — vẫn ≤ 2.
5. `"bbbbbab"` ⇒ **7**: cả chuỗi chỉ có `a, b`.

In [34]:
for s, expected in [("abaccc", 4), ("aabbcc", 4), ("abcabcabc", 2), ("Aa", 2), ("bbbbbab", 7)]:
    got = longest_two_distinct_sliding(s)
    assert got == expected, (s, got, expected)
    print(f"{s!r:12} -> {got}")

'abaccc'     -> 4
'aabbcc'     -> 4
'abcabcabc'  -> 2
'Aa'         -> 2
'bbbbbab'    -> 7


### Đáp án bài 2

| `right` | `ch` | `last` sau khi gán | 3 loại? | `victim` | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|---|
| 0 | a | {a:0} | không | | 0 | `a` | 1 |
| 1 | b | {a:0, b:1} | không | | 0 | `ab` | 2 |
| 2 | a | {a:2, b:1} | không | | 0 | `aba` | 3 |
| 3 | c | {a:2, b:1, c:3} | có | `b` (last=1) | 2 | `ac` | 3 |
| 4 | c | {a:2, c:4} | không | | 2 | `acc` | 3 |
| 5 | c | {a:2, c:5} | không | | 2 | `accc` | **4** |

Kiểm tra bằng `trace_last_index`:

In [35]:
assert trace_last_index("abaccc") == 4

right=0 'a' last={'a': 0} | cửa sổ 'a' | best=1
right=1 'b' last={'a': 0, 'b': 1} | cửa sổ 'ab' | best=2
right=2 'a' last={'a': 2, 'b': 1} | cửa sổ 'aba' | best=3
right=3 'c' last={'a': 2, 'b': 1, 'c': 3} -> victim='b', left=2 | cửa sổ 'ac' | best=3
right=4 'c' last={'a': 2, 'c': 4} | cửa sổ 'acc' | best=3
right=5 'c' last={'a': 2, 'c': 5} | cửa sổ 'accc' | best=4


### Đáp án bài 3 → 6

Giải thích bài 6 bằng tay với `"eceba"`:

- `atMost(2)`: theo bảng trace mục 8.1, độ dài cửa sổ hợp lệ tại `right = 0..4` là `1, 2, 3, 2, 2` ⇒ tổng **10** chuỗi con có ≤ 2 loại.
- `atMost(1)`: không có hai chữ giống nhau đứng liền nhau, nên mỗi cửa sổ chỉ dài 1 ⇒ `1 + 1 + 1 + 1 + 1` = **5** (chính là 5 chuỗi con dài 1).
- `exactly(2) = 10 - 5 = 5`: đó là `"ec"`, `"ce"`, `"eb"`, `"ba"`, `"ece"`.

Kiểm chứng:

In [36]:
def answer_longest_two_distinct_substring(s: str) -> str:
    count, left = {}, 0
    best_len, best_start = 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > 2:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        if right - left + 1 > best_len:
            best_len, best_start = right - left + 1, left
    return s[best_start:best_start + best_len]


def answer_longest_k_distinct_nonshrinking(s: str, k: int) -> int:
    count, left = {}, 0
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
        if len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
    return len(s) - left


class AnswerFruitSolution:
    def totalFruit(self, fruits: List[int]) -> int:
        last, left, best = {}, 0, 0
        for right, f in enumerate(fruits):
            last[f] = right
            if len(last) > 2:
                victim = min(last, key=last.get)
                left = last[victim] + 1
                del last[victim]
            best = max(best, right - left + 1)
        return best


def answer_count_at_most(s: str, k: int) -> int:
    count, left, total = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        total += right - left + 1          # số đoạn hợp lệ kết thúc tại right
    return total


def answer_count_exactly_two(s: str) -> int:
    return answer_count_at_most(s, 2) - answer_count_at_most(s, 1)


# Bài 3
assert answer_longest_two_distinct_substring("eceba") == "ece"
assert answer_longest_two_distinct_substring("ccaabbb") == "aabbb"
assert answer_longest_two_distinct_substring("aabbcc") == "aabb"
# Bài 4
assert answer_longest_k_distinct_nonshrinking("eceba", 2) == 3
assert answer_longest_k_distinct_nonshrinking("aa", 1) == 2
assert answer_longest_k_distinct_nonshrinking("abc", 0) == 0
assert answer_longest_k_distinct_nonshrinking("aabbcdef", 3) == 5
# Bài 5
assert AnswerFruitSolution().totalFruit([1, 2, 1]) == 3
assert AnswerFruitSolution().totalFruit([0, 1, 2, 2]) == 3
assert AnswerFruitSolution().totalFruit([1, 2, 3, 2, 2]) == 4
assert AnswerFruitSolution().totalFruit([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]) == 5
# Bài 6
assert answer_count_exactly_two("aab") == 2
assert answer_count_exactly_two("abc") == 2
assert answer_count_exactly_two("eceba") == 5


# Đối chiếu chéo với brute force trên input ngẫu nhiên
def brute_count_exactly_two(s):
    return sum(len(set(s[i:j + 1])) == 2 for i in range(len(s)) for j in range(i, len(s)))

def brute_longest_k(s, k):
    return max([j - i + 1 for i in range(len(s)) for j in range(i, len(s)) if len(set(s[i:j + 1])) <= k], default=0)

random.seed(340)
for _ in range(1000):
    s = "".join(random.choice("abcd") for _ in range(random.randint(1, 12)))
    k = random.randint(0, 4)
    assert answer_longest_k_distinct_nonshrinking(s, k) == brute_longest_k(s, k) == longest_k_distinct(s, k)
    assert answer_count_exactly_two(s) == brute_count_exactly_two(s)
    sub = answer_longest_two_distinct_substring(s)
    assert len(sub) == longest_two_distinct_bruteforce(s) and len(set(sub)) <= 2 and sub in s
    fruits = [ord(c) for c in s]
    assert AnswerFruitSolution().totalFruit(fruits) == longest_k_distinct(fruits, 2)

print("Đáp án bài 3–6: PASS toàn bộ test (kể cả 1000 bộ ngẫu nhiên)!")


print("atMost(2)  =", answer_count_at_most("eceba", 2))
print("atMost(1)  =", answer_count_at_most("eceba", 1))
print("exactly(2) =", answer_count_exactly_two("eceba"))
print("liệt kê   :", [ "eceba"[i:j + 1] for i in range(5) for j in range(i, 5) if len(set("eceba"[i:j + 1])) == 2])

Đáp án bài 3–6: PASS toàn bộ test (kể cả 1000 bộ ngẫu nhiên)!
atMost(2)  = 10
atMost(1)  = 5
exactly(2) = 5
liệt kê   : ['ec', 'ece', 'ce', 'eb', 'ba']


## 19. Template cuối cùng nên nhớ

```python
class Solution:
    def lengthOfLongestSubstringTwoDistinct(self, s: str) -> int:
        count, left, best = {}, 0, 0
        for right, ch in enumerate(s):
            count[ch] = count.get(ch, 0) + 1        # 1) thêm phải
            while len(count) > 2:                   # 2) vi phạm -> co trái
                out = s[left]
                count[out] -= 1
                if count[out] == 0:
                    del count[out]                  #    về 0 thì xóa key!
                left += 1
            best = max(best, right - left + 1)      # 3) ghi nhận
        return best
```

Câu thần chú: **"thêm phải — quá 2 thì co trái — về 0 thì xóa — ghi `right - left + 1`"**.

Hai biến thể nhớ kèm:

- **Vị trí cuối:** `last[ch] = right`; quá 2 loại ⇒ `victim = min(last, key=last.get)`, `left = last[victim] + 1`.
- **Không co:** đổi `while` thành `if`, bỏ `best`, trả `len(s) - left` (chỉ khi đề hỏi **độ dài**).

## 20. Checklist trước khi submit

- [ ] Đã dùng `dict` đếm (không phải `set`) cho cửa sổ trượt chưa?
- [ ] Khi đếm về `0` đã `del` key chưa? (`len(count)` phải bằng số loại thật)
- [ ] Thứ tự **thêm → co → ghi nhận** đúng chưa?
- [ ] Độ dài là `right - left + 1` (có `+ 1`) chưa?
- [ ] Nếu dùng Cách 3: `min(last, key=last.get)` (có `key=`) và `left = last[victim] + 1` chưa?
- [ ] Nếu dùng Cách 4: đề có thật sự chỉ hỏi **độ dài** không?
- [ ] Đã thử: 2 ví dụ đề, chuỗi 1 ký tự, chuỗi ≤ 2 loại (đáp án = `len(s)`), mọi ký tự khác nhau (đáp án 2), chữ hoa/thường, `"abaccc"`?
- [ ] Độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ — phù hợp `n = 10^5`.